In [218]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_magic_test.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_magic_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE5_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/XGB_stat_bin_median_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_combined_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/Cat_stat_median_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE5_test.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_stat_median_test.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_freq_test.csv
/kaggle/input/datasets/elenkapetrova/oofs12/Cat_TE_magic_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/XGB_best_test.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_combined_test.csv
/kaggle/input/datasets/elenkapetrova/oofs12/XGB_best_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/LGBM_freq_train.csv
/kaggle/input/datasets/elenkapetrova/oofs12/XGB_stat_median_test.c

# Import Libraries

In [219]:
from sklearn.linear_model import RidgeClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

import warnings
warnings.simplefilter("ignore")

# Data Loading

In [220]:
# Loading oofs
Df_summary=pd.read_csv("/kaggle/input/datasets/elenkapetrova/datasetoof/Df_summary.csv")
Df_summary_test=pd.read_csv("/kaggle/input/datasets/elenkapetrova/datasetoof/Df_summary_test.csv")  

In [221]:
#Loading oof-files

folder_path = '/kaggle/input/datasets/elenkapetrova/oofs12/'
all_files = os.listdir(folder_path)
all_files

for file in all_files:
    print("Loading file ", folder_path+file,"...")
    df_tmp = pd.read_csv(folder_path+file)
    if ("train" in file)==True:
        feature_name = file.replace("train.csv", "")
        for i in range(1,4):
             Df_summary[feature_name+str(i)] = df_tmp[feature_name+str(i)]
    if ("test" in file)==True:
        feature_name = file.replace("test.csv", "")
        for i in range(1,4):
             Df_summary_test[feature_name+str(i)] = df_tmp[feature_name+str(i)]
print("Loading is completed!")


Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_magic_test.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_magic_train.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE5_train.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/XGB_stat_bin_median_train.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE_combined_train.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/Cat_stat_median_train.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE5_test.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_stat_median_test.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_freq_test.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/Cat_TE_magic_train.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/XGB_best_test.csv ...
Loading file  /kaggle/input/datasets/elenkapetrova/oofs12/LGBM_TE

In [222]:
for df in [Df_summary, Df_summary_test]:
    df['Cat4']=np.argmax(df[['Cat4_1', 'Cat4_2', 'Cat4_3']], axis=1)
    df['XGB_OTE']=np.argmax(df[['XGB_OTE_1', 'XGB_OTE_2', 'XGB_OTE_3']], axis=1)
    df['Cat_OTE']=np.argmax(df[['Cat_OTE_1', 'Cat_OTE_2', 'Cat_OTE_3']], axis=1)
    df['LGBM_TE5']=np.argmax(df[['LGBM_TE5_1', 'LGBM_TE5_2', 'LGBM_TE5_3']], axis=1)
    df['XGB_stat_bin_median']=np.argmax(df[['XGB_stat_bin_median_1', 'XGB_stat_bin_median_2', 'XGB_stat_bin_median_3']], axis=1)    
    df['LGBM_TE_combined']=np.argmax(df[['LGBM_TE_combined_1', 'LGBM_TE_combined_2', 'LGBM_TE_combined_3']], axis=1)
    df['Cat_stat_median']=np.argmax(df[['Cat_stat_median_1', 'Cat_stat_median_2', 'Cat_stat_median_3']], axis=1)   
    df['LGBM_freq']=np.argmax(df[['LGBM_freq_1', 'LGBM_freq_2', 'LGBM_freq_3']], axis=1)
    df['Cat_stat_bin_median']=np.argmax(df[['Cat_stat_bin_median_1', 'Cat_stat_bin_median_2', 'Cat_stat_bin_median_3']], axis=1)
    df['LGBM_stat_median']=np.argmax(df[['LGBM_stat_median_1', 'LGBM_stat_median_2', 'LGBM_stat_median_3']], axis=1)
    df['LGBM_stat_bin_median']=np.argmax(df[['LGBM_stat_bin_median_1', 'LGBM_stat_bin_median_2', 'LGBM_stat_bin_median_3']], axis=1)
    df['XGB_stat_median']=np.argmax(df[['XGB_stat_median_1', 'XGB_stat_median_2', 'XGB_stat_median_3']], axis=1)
    df['XGB_best']=np.argmax(df[['XGB_best_1', 'XGB_best_2', 'XGB_best_3']], axis=1)

# Modeling

In [214]:
# set of features for metamodel 

features_list_to_metamodel=['LGBM1','LGBM1_1','LGBM1_2','LGBM1_3','LGBM2', 'LGBM2_1', 'LGBM2_2', 'LGBM2_3',
                            'Cat2', 'Cat2_1','Cat2_2','Cat2_3','XGB2','XGB2_1','XGB2_2','XGB2_3','XGB1',
                            'XGB1_1','XGB1_2','XGB1_3','Cat4_1','Cat4_2','Cat4_3','LGBM5', 'LGBM5_1',
                            'LGBM5_2','LGBM5_3','HGB2','HGB2_1','HGB2_2','HGB2_3','LGBM_TE1','LGBM_TE1_1',
                            'LGBM_TE1_2','LGBM_TE1_3','LGBM_TE4','LGBM_TE4_1','LGBM_TE4_2','LGBM_TE4_3',
                            'XGB_OTE_1','XGB_OTE_2','XGB_OTE_3','XGB_freq', 'XGB_freq_1', 'XGB_freq_2',
                            'XGB_freq_3','Cat_OTE_1','Cat_OTE_2','Cat_OTE_3','Log1','Log1_1','Log1_2',
                            'Log1_3','LGBM_TE_magic','LGBM_TE_magic_1','LGBM_TE_magic_2','LGBM_TE_magic_3',
                            'Cat_TE_magic','Cat_TE_magic_1','Cat_TE_magic_2','Cat_TE_magic_3','XGB_TE',
                            'XGB_TE_1','XGB_TE_2','XGB_TE_3','MLP_TE','MLP_TE_1','MLP_TE_2','MLP_TE_3',
                            'LGBM_stat', 'LGBM_stat_1','LGBM_stat_2','LGBM_stat_3','LGBM_TE5_1','LGBM_TE5_2',
                            'LGBM_TE5_3','XGB_stat_bin_median_1','XGB_stat_bin_median_2','XGB_stat_bin_median_3',
                            'LGBM_TE_combined_1','LGBM_TE_combined_2','LGBM_TE_combined_3','Cat_stat_median_1',
                            'Cat_stat_median_2','Cat_stat_median_3', 'LGBM_freq_1','LGBM_freq_2','LGBM_freq_3',
                            'Cat_stat_bin_median_1', 'Cat_stat_bin_median_2', 'Cat_stat_bin_median_3','LGBM_stat_median_1',
                            'LGBM_stat_median_2','LGBM_stat_median_3','LGBM_stat_bin_median_1','LGBM_stat_bin_median_2',
                            'LGBM_stat_bin_median_3','XGB_stat_median_1','XGB_stat_median_2', 'XGB_stat_median_3',
                            'Cat4', 'XGB_OTE','Cat_OTE','LGBM_TE5','XGB_stat_bin_median','LGBM_TE_combined',
                            'Cat_stat_median','LGBM_freq','Cat_stat_bin_median','LGBM_stat_median','LGBM_stat_bin_median',
                            'XGB_stat_median','XGB_best_1','XGB_best_2','XGB_best_3', 'XGB_best',  'Orig_Low','Orig_Medium',
                             'Orig_High',]

In [227]:
# metamodel's parameters have been tuned with optuna 

metamodel=RidgeClassifier(random_state=100, alpha= 16.805290034232804,  
                          class_weight=   {0:  1.445925258585079, 1: 30.956581677130455, 2:   2.5279265602123617})


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 )

oof_pred=np.zeros(Df_summary.shape[0], dtype=float)
y_pred=np.zeros(Df_summary_test.shape[0], dtype=int)

for train_index, test_index in kf.split(Df_summary, Df_summary.Irrigation_Need):
    X_train_cv, X_test_cv = Df_summary.iloc[train_index],Df_summary.iloc[test_index]
    y_train_cv, y_test_cv =  Df_summary.iloc[train_index].Irrigation_Need ,Df_summary.iloc[test_index].Irrigation_Need
    

    metamodel.fit(X_train_cv[features_list_to_metamodel], y_train_cv)




    oof_pred[test_index]= metamodel.predict(X_test_cv[features_list_to_metamodel])
 

BAC =balanced_accuracy_score(Df_summary.Irrigation_Need, oof_pred)
print("Balanced_accuracy_score on CV = ", BAC)

Balanced_accuracy_score on CV =  0.9806888171324801


In [224]:
metamodel.fit(Df_summary[features_list_to_metamodel], Df_summary.Irrigation_Need)
Df_summary_test["Irrigation_Need"] = metamodel.predict(Df_summary_test[features_list_to_metamodel])

 

Df_summary_test["Irrigation_Need_Str"]=Df_summary_test.Irrigation_Need.apply(lambda x : "Low" if x==0 else ( "High" if x==1 else "Medium"))

In [225]:
df_result = pd.concat([Df_summary_test["id"].reset_index(), pd.DataFrame(Df_summary_test["Irrigation_Need_Str"]).reset_index()], axis=1)
df_result=df_result.drop("index", axis=1)
df_result.to_csv("sample_submission.csv", index=False)